# Notebook 03 -- Champion Model: Hybrid AO-PSO Tuned TFT

**Research:** Hybrid AO-PSO Optimised Temporal Fusion Transformer for Smart Grid Multi-Horizon Energy Forecasting

---

## Purpose

This notebook evaluates the **Champion Algorithm** -- the Hybrid AO-PSO cascade -- across the v2.1 Protocol (7 runs x 30 iterations) for both datasets. The results produced here must demonstrate superiority over:

- **Notebook 01 baselines:** Standard LSTM, Bi-LSTM, XGBoost (forecasting accuracy)
- **Notebook 02 standalone swarms:** Pure AO, Pure PSO (convergence speed and accuracy)

## The Hybrid Cascade

The Hybrid AO-PSO operates in two sequential phases within a single optimization run:

| Phase | Algorithm | Iterations | Strategy |
|-------|-----------|------------|----------|
| 1: Global Exploration | Aquila Optimizer | 1 -- 15 (50%) | 4-phase hunting: high soar, contour flight, low flight, walk/grab |
| 2: Local Exploitation | Particle Swarm | 16 -- 30 (50%) | Velocity-position updates with cognitive/social components |

At the **hand-off point** (iteration 15), the AO's best-discovered population positions and global best are injected into a fresh PSO swarm via `_inject_swarm_state()`. This gives PSO a warm-started, globally-informed search space to exploit locally, combining the strengths of both algorithms.

**Thesis claim:** This two-phase cascade achieves optimal hyperparameter convergence significantly faster than either algorithm in isolation, while producing equal or better final forecasting accuracy.

**v2.1 Methodology Note:** A pilot study (50 iterations, pop=20) confirmed convergence of best fitness by iteration 6, with swarm mean stabilizing by iteration 29. Runs are capped at 30 iterations. All 7 champion configurations are trained + evaluated on full data to demonstrate optimizer reliability (not just single-point evidence).

Results (metrics, discovered hyperparameters, and full convergence histories with phase labels) are saved to `results/hybrid_metrics.json` for the Wilcoxon tests and convergence plots in Notebook 04.

In [ ]:
# ================================================================
# Cell 1: Imports
# ================================================================
import sys
import os
import json
import time
import gc
import logging
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

# ---- Set project root ----
PROJECT_ROOT = "/workspace/Hybrid-Optimizer-AO-PSO"

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src")), (
    f"src/ not found in {PROJECT_ROOT}. "
    f"Update PROJECT_ROOT to your actual project path on this machine."
)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

from src.dataset_utils import prepare_pipeline
from src.models import BaseTFT
from src.optimizers import ObjectiveFunction, Hybrid_AO_PSO
from src.metrics import (
    inverse_transform_predictions,
    calculate_forecasting_metrics,
    calculate_horizon_metrics,
)

# Enable optimizer logging so the AO -> PSO hand-off is visible
logging.basicConfig(level=logging.INFO, format="%(message)s")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project root : {PROJECT_ROOT}")
print(f"Device       : {DEVICE}")
print(f"PyTorch      : {torch.__version__}")

In [ ]:
# ================================================================
# Cell 2: Configuration & Hyperparameters
# ================================================================
# All settings match Notebook 02 for a fair comparison,
# with the addition of AO_FRACTION for the hybrid cascade.

# --- v2.1 Protocol (pilot-validated convergence cutoff) ---
N_RUNS = 7              # 7 runs for Wilcoxon power (min p≈0.008 one-tailed)

# --- Hybrid AO-PSO configuration ---
POP_SIZE    = 20        # Swarm population size
TOTAL_ITER  = 30        # v2.1: 30 iter (was 50) -- pilot confirmed plateau by iter 6
AO_FRACTION = 0.5       # 50% AO (global) -> 50% PSO (local)

# --- Proxy fitness settings (identical to Notebook 02) ---
PROXY_EPOCHS    = 2
SUBSET_FRACTION = 0.3

# --- Full training after optimization (identical to Notebook 02) ---
FULL_EPOCHS = 30
PATIENCE    = 5

# --- Data pipeline ---
WINDOW_SIZE = 168       # 7 days of hourly data
HORIZON     = 24        # 24-hour forecast
BATCH_SIZE  = 64
NUM_WORKERS = os.cpu_count()  # Use all available CPU cores for data loading

# --- Dataset paths ---
MICRO_PATH = "data/micro_household_dataset.csv"
MACRO_PATH = "data/macro_grid_dataset.csv"

# --- Results directory ---
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print("Configuration (v2.1 Protocol)")
print("=" * 40)
print(f"  N_RUNS          : {N_RUNS}")
print(f"  POP_SIZE        : {POP_SIZE}")
print(f"  TOTAL_ITER      : {TOTAL_ITER}")
print(f"  AO_FRACTION     : {AO_FRACTION}")
print(f"  AO iterations   : {int(TOTAL_ITER * AO_FRACTION)}")
print(f"  PSO iterations  : {TOTAL_ITER - int(TOTAL_ITER * AO_FRACTION)}")
print(f"  PROXY_EPOCHS    : {PROXY_EPOCHS}")
print(f"  SUBSET_FRACTION : {SUBSET_FRACTION}")
print(f"  FULL_EPOCHS     : {FULL_EPOCHS}")
print(f"  PATIENCE        : {PATIENCE}")
print(f"  WINDOW_SIZE     : {WINDOW_SIZE}")
print(f"  HORIZON         : {HORIZON}")
print(f"  NUM_WORKERS     : {NUM_WORKERS}")
print(f"  Results dir     : {RESULTS_DIR.resolve()}")

In [ ]:
# ================================================================
# Cell 3: Hybrid Evaluation Wrapper
# ================================================================

def train_tft_full(model, train_loader, val_loader, epochs, lr, device):
    """
    Train a BaseTFT model for the full epoch budget with early stopping.
    Identical to Notebook 02 for experimental consistency.
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    patience_counter = 0
    best_state = None

    for epoch in range(1, epochs + 1):
        # --- Train ---
        model.train()
        for x_cont, x_cat, y in train_loader:
            x_cont, y = x_cont.to(device), y.to(device)
            optimizer.zero_grad()
            preds = model(x_cont)
            loss = criterion(preds, y)
            loss.backward()
            optimizer.step()

        # --- Validate ---
        model.eval()
        val_losses = []
        with torch.no_grad():
            for x_cont, x_cat, y in val_loader:
                x_cont, y = x_cont.to(device), y.to(device)
                preds = model(x_cont)
                val_losses.append(criterion(preds, y).item())
        val_loss = np.mean(val_losses)

        # --- Early stopping ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_tft(model, test_loader, device):
    """Run inference on the test set, return (preds, targets) as numpy."""
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for x_cont, x_cat, y in test_loader:
            x_cont = x_cont.to(device)
            preds = model(x_cont).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(y.numpy())
    return np.concatenate(all_preds), np.concatenate(all_targets)


def evaluate_hybrid_swarm(pipeline, n_runs):
    """
    Run the full 10x Grind for the Hybrid AO-PSO + BaseTFT.

    For each run:
      1. Instantiate ObjectiveFunction with proxy settings.
      2. Run Hybrid_AO_PSO to discover best_params.
      3. Train a fresh BaseTFT using best_params for FULL_EPOCHS.
      4. Evaluate on the test set with inverse-transform + metrics.

    The convergence history (with 'phase' labels: 'AO' / 'PSO') is
    saved per-run so Notebook 04 can plot the hand-off curves.

    Parameters
    ----------
    pipeline : dict
        Output of prepare_pipeline().
    n_runs : int
        Number of independent runs.

    Returns
    -------
    list[dict]
        Per-run results with metrics, best_params, convergence, and timing.
    """
    train_loader = pipeline["train_loader"]
    val_loader   = pipeline["val_loader"]
    test_loader  = pipeline["test_loader"]
    scaler       = pipeline["scaler"]
    target_idx   = pipeline["target_idx"]
    n_features   = pipeline["n_continuous"]

    results = []

    for run in range(1, n_runs + 1):
        print(f"\n{'=' * 60}")
        print(f"  [Hybrid AO-PSO] Run {run}/{n_runs}")
        print(f"{'=' * 60}")
        t0 = time.time()

        # ----- Phase 1+2: Hybrid Optimization -----
        obj_fn = ObjectiveFunction(
            train_loader=train_loader,
            val_loader=val_loader,
            n_encoder_features=n_features,
            window_size=WINDOW_SIZE,
            horizon=HORIZON,
            proxy_epochs=PROXY_EPOCHS,
            subset_fraction=SUBSET_FRACTION,
            device=DEVICE,
        )

        hybrid = Hybrid_AO_PSO(
            objective_fn=obj_fn,
            pop_size=POP_SIZE,
            total_iter=TOTAL_ITER,
            ao_fraction=AO_FRACTION,
            seed=run,  # Different seed each run for independence
        )

        best_params, best_fitness, convergence = hybrid.optimize()
        opt_time = time.time() - t0

        # Report the hand-off point
        ao_iters = [r["iteration"] for r in convergence.history
                    if r.get("phase") == "AO"]
        pso_iters = [r["iteration"] for r in convergence.history
                     if r.get("phase") == "PSO"]
        print(f"  Optimization complete in {opt_time:.1f}s")
        print(f"  AO iterations  : {len(ao_iters)}")
        print(f"  PSO iterations : {len(pso_iters)}")
        print(f"  Proxy Val RMSE : {best_fitness:.6f}")
        print(f"  Discovered hyperparameters:")
        for k, v in best_params.items():
            print(f"    {k:22s}: {v}")

        # ----- Full Training with best_params -----
        print(f"  Training fresh BaseTFT for {FULL_EPOCHS} epochs...")
        t1 = time.time()

        model = BaseTFT(
            n_encoder_features=n_features,
            n_decoder_features=0,
            n_static_categoricals=0,
            d_model=best_params["d_model"],
            n_heads=best_params["n_heads"],
            num_encoder_layers=best_params["num_encoder_layers"],
            dropout=best_params["dropout"],
            horizon=HORIZON,
            window_size=WINDOW_SIZE,
        )

        model = train_tft_full(
            model, train_loader, val_loader,
            epochs=FULL_EPOCHS,
            lr=best_params["learning_rate"],
            device=DEVICE,
        )
        train_time = time.time() - t1

        # ----- Test Evaluation -----
        preds_scaled, targets_scaled = predict_tft(model, test_loader, DEVICE)

        preds_real, targets_real = inverse_transform_predictions(
            preds_scaled, targets_scaled, scaler, target_idx,
        )

        overall = calculate_forecasting_metrics(targets_real, preds_real)
        per_horizon = calculate_horizon_metrics(targets_real, preds_real)

        total_time = time.time() - t0

        # ----- Serialize convergence history (with phase labels) -----
        convergence_data = convergence.to_dict()

        run_result = {
            "run":            run,
            "RMSE":           overall["RMSE"],
            "MAE":            overall["MAE"],
            "MAPE":           overall["MAPE"],
            "horizon_metrics": {
                str(h): m for h, m in per_horizon.items()
            },
            "best_params":    best_params,
            "proxy_fitness":  best_fitness,
            "convergence":    convergence_data,
            "opt_time_s":     round(opt_time, 2),
            "train_time_s":   round(train_time, 2),
            "total_time_s":   round(total_time, 2),
        }
        results.append(run_result)

        print(f"\n  [Hybrid AO-PSO] Run {run:2d}/{n_runs} COMPLETE")
        print(f"    Test RMSE : {overall['RMSE']:.4f}")
        print(f"    Test MAE  : {overall['MAE']:.4f}")
        print(f"    Test MAPE : {overall['MAPE']:.2f}%")
        print(f"    Opt time  : {opt_time:.1f}s | Train time: {train_time:.1f}s | Total: {total_time:.1f}s")

        # ----- Memory cleanup -----
        del model, hybrid, obj_fn, convergence
        torch.cuda.empty_cache()
        gc.collect()

    return results


def print_summary(label, results):
    """Print Mean +/- Std for RMSE, MAE, MAPE across all runs."""
    rmses = [r["RMSE"] for r in results]
    maes  = [r["MAE"]  for r in results]
    mapes = [r["MAPE"] for r in results]
    times = [r["total_time_s"] for r in results]
    print(f"\n{'=' * 60}")
    print(f"  {label} -- {len(results)} runs")
    print(f"{'=' * 60}")
    print(f"  RMSE : {np.mean(rmses):.4f} \u00b1 {np.std(rmses):.4f}")
    print(f"  MAE  : {np.mean(maes):.4f} \u00b1 {np.std(maes):.4f}")
    print(f"  MAPE : {np.mean(mapes):.2f}% \u00b1 {np.std(mapes):.2f}%")
    print(f"  Time : {np.mean(times):.1f}s \u00b1 {np.std(times):.1f}s per run")
    print(f"{'=' * 60}")


print("Evaluation functions defined.")

In [ ]:
# ================================================================
# Cell 4: Micro-Grid Execution
# ================================================================
print("Loading Micro-Grid dataset (UCI Household)...")
micro_pipeline = prepare_pipeline(
    filepath=MICRO_PATH,
    dataset_type="micro",
    window_size=WINDOW_SIZE,
    horizon=HORIZON,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)
print(f"  n_continuous : {micro_pipeline['n_continuous']}")
print(f"  target_col   : {micro_pipeline['target_col']}")
print(f"  train batches: {len(micro_pipeline['train_loader'])}")
print(f"  test  batches: {len(micro_pipeline['test_loader'])}")

print("\n" + "#" * 60)
print("#  Hybrid AO-PSO (Micro-Grid) -- 7x Grind (v2.1)")
print("#" * 60)
micro_hybrid = evaluate_hybrid_swarm(micro_pipeline, N_RUNS)
print_summary("Hybrid AO-PSO-TFT (Micro)", micro_hybrid)

In [ ]:
# ================================================================
# Cell 5: Macro-Grid Execution
# ================================================================
print("Loading Macro-Grid dataset (ISO-NE)...")
macro_pipeline = prepare_pipeline(
    filepath=MACRO_PATH,
    dataset_type="macro",
    window_size=WINDOW_SIZE,
    horizon=HORIZON,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)
print(f"  n_continuous : {macro_pipeline['n_continuous']}")
print(f"  target_col   : {macro_pipeline['target_col']}")
print(f"  train batches: {len(macro_pipeline['train_loader'])}")
print(f"  test  batches: {len(macro_pipeline['test_loader'])}")

print("\n" + "#" * 60)
print("#  Hybrid AO-PSO (Macro-Grid) -- 7x Grind (v2.1)")
print("#" * 60)
macro_hybrid = evaluate_hybrid_swarm(macro_pipeline, N_RUNS)
print_summary("Hybrid AO-PSO-TFT (Macro)", macro_hybrid)

In [ ]:
# ================================================================
# Cell 6: Save Results
# ================================================================
# Structure: { dataset -> [ {run metrics + convergence} ] }
# Notebook 04 will load this alongside baseline_metrics.json and
# standalone_swarm_metrics.json to run Wilcoxon tests and plot
# the AO -> PSO hand-off convergence curves.

all_results = {
    "micro": {
        "Hybrid_AO_PSO": micro_hybrid,
    },
    "macro": {
        "Hybrid_AO_PSO": macro_hybrid,
    },
    "config": {
        "n_runs":          N_RUNS,
        "pop_size":        POP_SIZE,
        "total_iter":      TOTAL_ITER,
        "ao_fraction":     AO_FRACTION,
        "proxy_epochs":    PROXY_EPOCHS,
        "subset_fraction": SUBSET_FRACTION,
        "full_epochs":     FULL_EPOCHS,
        "patience":        PATIENCE,
        "window_size":     WINDOW_SIZE,
        "horizon":         HORIZON,
        "batch_size":      BATCH_SIZE,
    },
}

output_path = RESULTS_DIR / "hybrid_metrics.json"
with open(output_path, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"Results saved to: {output_path.resolve()}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")
print("\n>>> CHAMPION HYBRID TESTING COMPLETE <<<")